In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from mdbmaster import MasterParams
from sys import prefix
mp = MasterParams(verbose=True)
io = FileIO()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from mdblib import spotify
mio   = spotify.MusicDBIO(verbose=True)
apiio = spotify.RawAPIData()
db    = mio.db

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb
MusicDBBaseDirs(db=Spotify)
   RawDataDir     = /Volumes/Piggy/Discog/artists-spotify
   ModValDataDir  = /Volumes/Seagate/Discog/artists-spotify-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-spotify-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-spotify
ParseRawDataUtils(mdbdata, mdbdir) [Spotify]
Spotify ModValMetaData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Link
  ==> Metric
  ==> Counts


# Perm Dir

In [4]:
def setupPermDir(db):
    mp = MasterParams(verbose=False)
    prefixDir = DirInfo(prefix)
    projDir   = prefixDir.join(mp.getProjectName())
    projDir.mkDir()
    libDir    = projDir.join("mdblib")
    libDir.mkDir()
    permDBDir = libDir.join(db)
    permDBDir.mkDir()
    return permDBDir
permDBDir = setupPermDir(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Spotify DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Spotify


# Master Files

In [5]:
from mdbbase import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistIDs     = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDs".format(db.lower()))
localArtistIDsData = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDsData".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
knownArtists       = mio.data.getArtistIDToNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist IDs:          {0}".format(len(localArtistIDs.get())))
print("   Local Artist IDs Data:     {0}".format(len(localArtistIDsData.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

Spotify Search Results
   Global Master Search:      123034
   Global Master Search Data: 0
   Local Artist Search:       530953
   Local Artist IDs:          43558
   Local Artist IDs Data:     0
   Local Album Search:        126155
   Errors:                    31
   Known Summary IDs:         662914


# Download Artist Search Data

In [7]:
mio   = spotify.MusicDBIO(verbose=False)
apiio = spotify.RawAPIData(debug=True)

Spotify API(Key={'client_id': '61e441c3b90c4873aa0e6b9582564f95', 'client_secret': 'ae0d0f968bf443fdac1d9ac6ef65fc0f'})


In [9]:
from musicdb import MusicDBIO
from pandas import Series
mdbio = MusicDBIO()
mmeDF = mdbio.getData()
unmatchedArtistNames = mmeDF[mmeDF["Spotify"].isna()]["ArtistName"]
searchedForArtistsSeries = Series(masterArtists.get())
artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForArtistsSeries.index)]

print("{0} Search Results".format(db))
print("   Known Master Artist Names: {0}".format(len(unmatchedArtistNames)))
print("   Known Local Artist Names:  {0}".format(len(searchedForArtistsSeries)))
print("   Artist Names To Get:       {0}".format(len(artistNamesToGet)))

Spotify Search Results
   Known Master Artist Names: 757631
   Known Local Artist Names:  123034
   Artist Names To Get:       626393


In [ ]:
saveSearchedForMasterArtistsData({})

In [13]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "9:15pm")
tt = TermTime("today", "3:15pm")
n  = 0
searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if searchedForMasterArtists.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForMasterArtists[artistName] = True
        apiio.sleep(3.5)
        continue
        
    searchedForMasterArtistsData[artistName] = response
    searchedForMasterArtists[artistName] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)

Starting Process [Getting Spotify ArtistIDs]    ==> Time Is 2022-03-04 13:50:14
========================= termTime(day=today,time=3:15pm) =========================
   ====> Terminate Time Set To 2022-03-04 15:15:00 <====
   ====> Will Terminate Process 1 Hour and 24 Minutes From Now
Searching For Nadeah                                            2
Searching For Ming Tsao                                         1
Searching For Geyers                                            2
Searching For Hannes Schweiger                                  0
Error ==> ('hhhhhhhhXXX0002763XXX1', 'Hannes Schweiger')
Searching For Neil Yates                                        3
Searching For Woodworkings                                      1
Searching For Сонцесвiт                                         2
Searching For Janne Mark                                        3
Searching For Dario Bruna                                       3
Searching For Dietrich Jeske                                    0

Searching For Fifou                                             17
Searching For Little Joe                                        50
Searching For Ollabelle                                         1
Searching For Musica Elettronica Viva                           1
Searching For Bill Goodwin                                      4
Searching For Laza Ristovski                                    2
Searching For Lena Agree                                        0
Error ==> ('llllllllXXX0010827XXX1', 'Lena Agree')
Searching For Māris Sirmais                                    2
Searching For Alexis Descharmes                                 1
Searching For Jan Hellriegel                                    1
Searching For Ernie Rettino                                     1
Searching For David Mansfield                                   2
Searching For Nadir Khayat                                      2
Searching For Trespassers W                                     3
Searching For Joe Moody

Searching For Ash Borer                                         1
Searching For Mazut                                             3
Searching For Andris Mattson                                    1
Searching For Donna Coleman                                     1
Searching For Dave "O" Ogrin                                    1
Searching For Jim Boyd                                          11
Searching For Ilene Barnes                                      1
Searching For Windows                                           50
Searching For Eon                                               50
225/?      : Process [Getting Spotify ArtistIDs] Has Run For 16 Minutes and 28 Seconds.
Saving 121889 Spotify Searched For Artist IDs
Searching For Diskoteka Avaria                                  1
Searching For Nuit Noire                                        3
Searching For Aisha Kahlil                                      1
Searching For Bennie Crawford                                   0
Error

Searching For Olaf Ott                                          4
Searching For Andrew McGee                                      2
Searching For Christian Leitgeb                                 0
Error ==> ('ccccccccXXX0020658XXX1', 'Christian Leitgeb')
Searching For ADL                                               50
Searching For Raimondo Caruana                                  0
Error ==> ('rrrrrrrrXXX0002502XXX1', 'Raimondo Caruana')
Searching For Dirk Löwenhaupt                                  0
Error ==> ('ddddddddXXX0031359XXX1', 'Dirk Löwenhaupt')
Searching For Brian Holland                                     8
Searching For Sora Naomi                                        1
Searching For Mexicali Brass                                    1
Searching For Joe Lemon                                         10
Searching For Cakra Khan                                        3
Searching For Dominic Behan                                     2
Searching For Wade Walston        

Searching For Jaws                                              50
Searching For Benümb                                           2
Searching For Steven Mayer                                      2
Searching For Sleep ∞ Over                                      ==> Error in Spotify search for Sleep ∞ Over
Error ==> ('ssssssssXXX0030651XXX1', 'Sleep ∞ Over')
Searching For Ian Fountain                                      1
Searching For Le Forte Four                                     1
Searching For 3 Leg Torso                                       3
Searching For Preacher Jack                                     1
Searching For Hesaäijä                                        1
Searching For Kevin Paige                                       1
Searching For Jurica Murai                                      2
Searching For Albert Malawi                                     1
Searching For Juan Martinez                                     50
Searching For Véronique Fèvre             

Searching For Steve 'n' Seagulls                                1
Searching For Eloquence                                         22
Searching For Jason Hellmann                                    0
Error ==> ('jjjjjjjjXXX0011750XXX1', 'Jason Hellmann')
Searching For Roy DeCarava                                      0
Error ==> ('rrrrrrrrXXX0031905XXX1', 'Roy DeCarava')
Searching For Carolina Slim                                     8
Searching For Greg Cohen                                        5
Searching For Carlos LeBron                                     0
Error ==> ('ccccccccXXX0005396XXX1', 'Carlos LeBron')
Searching For Dani Moreno                                       19
Searching For Hip                                               50
Searching For Alan Ravenscroft                                  0
Error ==> ('aaaaaaaaXXX0013058XXX1', 'Alan Ravenscroft')
Searching For T-BOLAN                                           21
Searching For Rick Wilhite                         

Searching For Rob Piltch                                        2
Searching For Richard Hynd                                      5
Searching For Gary Martin                                       10
Searching For Carter CC Colins                                  0
Error ==> ('ccccccccXXX0007125XXX1', 'Carter CC Colins')
Searching For A*S*Y*S                                           50
Searching For Koeckert-Quartett                                 1
Searching For Chris Weber                                       16
Searching For Gus Edwards                                       6
Searching For John Ellison                                      7
Searching For Gordon Beecher                                    0
Error ==> ('ggggggggXXX0017853XXX1', 'Gordon Beecher')
Searching For Giovanni Foiani                                   3
Searching For Vincent Niclo                                     2
Searching For Linda Draper                                      1
Searching For illScarlett  

Searching For Derek Douget                                      1
Searching For Pip Brown                                         4
Searching For The Lovetones                                     5
Searching For Kimiko Whittaker                                  0
Error ==> ('kkkkkkkkXXX0016381XXX1', 'Kimiko Whittaker')
Searching For Safi Connection                                   5
Searching For Bryan K. Christner                                0
Error ==> ('bbbbbbbbXXX0036485XXX1', 'Bryan K. Christner')
Searching For Bent Jædig                                        3
Searching For Cloë Elmo                                        2
Searching For Swantje Hoffmann                                  1
Searching For Luca Marzana                                      1
Searching For Lee Shapiro                                       2
Searching For Thierry Gotti                                     2
Searching For Boncana Maïga                                    2
Searching For Victor Babin

Searching For Volcano Choir                                     1
Searching For Dimelo Flow                                       2
Searching For Joseph Hammer                                     2
Searching For Passe-Partout                                     7
Searching For Craig Hooper                                      0
Error ==> ('ccccccccXXX0036042XXX1', 'Craig Hooper')
Searching For L-P Anderson                                      34
Searching For Kim Deschamps                                     2
Searching For Just Loomis                                       0
Error ==> ('jjjjjjjjXXX0050219XXX1', 'Just Loomis')
Searching For Ricky Andreoni                                    0
Error ==> ('rrrrrrrrXXX0017867XXX1', 'Ricky Andreoni')
Searching For Shaun Bloodworth                                  0
Error ==> ('ssssssssXXX0019480XXX1', 'Shaun Bloodworth')
Searching For Palm Skin Productions                             1
Searching For Timex Social Club                         

Searching For Másfél                                          1
Searching For Raimo Kangro                                      1
Searching For Ava Cherry                                        3
Searching For Dino J.A. Deane                                   3
1000/?     : Process [Getting Spotify ArtistIDs] Has Run For 1 Hour and 19 Minutes.
Saving 122939 Spotify Searched For Artist IDs
Searching For Robyn Archer                                      5
Searching For Dave Kirby                                        2
Searching For Benoît Alziary                                   0
Error ==> ('bbbbbbbbXXX0011639XXX1', 'Benoît Alziary')
Searching For Stefano Pastor                                    4
Searching For Scott Bender                                      2
Searching For Charlie Spand                                     2
Searching For Zach Brock                                        3
Searching For Pierre Flynn                                      1
Searching For Joe Shay

In [10]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = DataFrame(getFlatList([v for k,v in mad.items()]))
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [11]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = spotify.MusicDBIO(verbose=False,local=False).data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    spotify.MusicDBIO(verbose=False,local=False).data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

Found 8031 New Artists
Found 662647 Previous Artists
Found 670678 Total Artists
Found 662647 Unique Artists
Found 662647 Unique + Sorted Artists
Saving Data


In [13]:
masterArtistsData.save(data={})

# Download Data By Artist IDs

In [ ]:
mio   = spotify.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = spotify.RawAPIData(debug=True)

## Find Artists To Download

In [18]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(localArtistIDs.get().keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

Spotify Search Results
   Known Summary IDs:    662914
   Artists IDs To Get:   267
   Artists IDs To Get:   0


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = localArtistIDsData.get()
searchedForArtistIDs     = localArtistIDs.get()
searchedForErrors        = errors.get()
for i,dbID in enumerate(artistIDsToGet):
    if searchedForArtistIDs.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        localArtistIDs.save(data=searchedForArtistIDs)
        localArtistIDsData.save(data=searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
localArtistIDs.save(data=searchedForArtistIDs)
localArtistIDsData.save(data=searchedForArtistIDsData)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

## Move Local

In [ ]:
from mdblib.spotify import moveLocalFiles

In [ ]:
moveLocalFiles()

# Download Album Data

In [7]:
mio   = spotify.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = spotify.RawAPIData(debug=True)

Spotify API(Key={'client_id': '61e441c3b90c4873aa0e6b9582564f95', 'client_secret': 'ae0d0f968bf443fdac1d9ac6ef65fc0f'})


## Find Albums To Get

In [14]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(localAlbums.get().keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

Spotify Search Results
   Known Summary IDs:    662914
   Artists IDs To Get:   523764


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "4:00pm")
#tt = TermTime("today", "9:15pm")
#tt = TermTime("today", "11:00am")
n  = 0
searchedForAlbums = localAlbums.get()
searchedForErrors = errors.get()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        localAlbums.save(data=searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
localAlbums.save(data=searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Starting Process [Getting Spotify Albums]    ==> Time Is 2022-03-07 11:10:49
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-03-08 11:00:00 <====
   ====> Will Terminate Process 23 Hours and 49 Minutes From Now
Searching For Albums For DopeBoi O'ganja (6eqzkM2QHXVnnSJSFE3Rb2)          	   ===> [4]        4  4
Searching For Albums For DrekkaFineAss (2qH1QZ6vAwLNTmMqyjiQTz)            	   ===> [1]        1  1
Searching For Albums For Filarmonica Vincenzo Bellini (3IzYkXqpgaO5nbEhN6XzlD)	   ===> [2]        2  2
Searching For Albums For Versailles Guitar Quartet (3P6lcl0s9ugcV9rkQKb64N)	   ===> [1]        1  1
Searching For Albums For Eskmo (6zfCsI5zOpdtuZOG5k3nI8)                    	   ===> [1]        1  1
5/?        : Process [Getting Spotify Albums] Has Run For 47 Seconds.
Saving 139155 Spotify Searched For Albums
Searching For Albums For PACIFICO RECORDS (1uBRPRGIWqKrLMXrk1AUe2)         	   ===> [1]        1  

45/?       : Process [Getting Spotify Albums] Has Run For 8 Minutes and 29 Seconds.
Saving 139195 Spotify Searched For Albums
Searching For Albums For Pierrot Mbiya Blessings (4OTRQkVOMaxqNmTwE50Wyk)  	   ===> [1]        1  1
Searching For Albums For Pierrot panse (0DdVWQNPsAQbybEMlbgubS)            	   ===> [2]        2  2
Searching For Albums For Beat Widmann Gen. Machinger (ca. 1476-1551) (3MYefAM8lBwtFMh8g91UL0)	   ===> [1]        1  1
Searching For Albums For Paul Verlaine (0hXLDvT3t2wse1UfKKahuC)            	   ===> [2]        2  2
Searching For Albums For Golf Uniforme (79Xn5CSyO4H1vcyP47aQsp)            	   ===> [1]        1  1
50/?       : Process [Getting Spotify Albums] Has Run For 9 Minutes and 23 Seconds.
Saving 139200 Spotify Searched For Albums
Searching For Albums For The Unknown (5j0y3rZ5gaUIPSbaScvGDd)              	   ===> [1]        1  1
Searching For Albums For Outragers (7vypZiteIqs8h2VNjgrHU7)                	   ===> [3]        3  3
Searching For Albums For Valky

Searching For Albums For Universe (0GYrTCFyPao1AmaxFZNTkz)                 	   ===> [1]        1  1
Searching For Albums For igi Youngbull (0g8GeP3ffGVaANeqNnV3nX)            	   ===> [1]        1  1
90/?       : Process [Getting Spotify Albums] Has Run For 17 Minutes and 35 Seconds.
Saving 139240 Spotify Searched For Albums
Searching For Albums For Eddie Cain featuring Script of The Outsidaz (2EwvFsa4oxWDMeWTGnGxHB)	   ===> [1]        1  1
Searching For Albums For Lokassa Ya Mbongo and Sam Mangwana (4aHspmXH8KIBXwuo4ktWJN)	   ===> [1]        1  1
Searching For Albums For Samuel Jones (5hBf76KWUNJVAijOjlFD0y)             	   ===> [1]        1  1
Searching For Albums For Sam Roberts & Ministry (0N8GvGhEemOXsZG2nHLcCM)   	   ===> [1]        1  1
Searching For Albums For Van Goghst (4UNoPlCbizpEFSeaCRVDzN)               	   ===> [1]        1  1
95/?       : Process [Getting Spotify Albums] Has Run For 18 Minutes and 29 Seconds.
Saving 139245 Spotify Searched For Albums
Searching For Album

Searching For Albums For Tim Hagans & Norbotten Big Band (5Cd0v2fBAKOx3FgkqeLkck)	   ===> [1]        1  1
Searching For Albums For The Whites of Texas (2h6zNLC3yXWAgYD4dS2cnA)      	   ===> [2]        2  2
Searching For Albums For Tim Garland Quartet (2hnPnFQUNoJdFC5rFYtv1O)      	   ===> [3]        3  3
Searching For Albums For John Wirtz (2YgCw2r3pTVaV3qgaihmr9)               	   ===> [1]        1  1
135/?      : Process [Getting Spotify Albums] Has Run For 26 Minutes and 42 Seconds.
Saving 139285 Spotify Searched For Albums
Searching For Albums For Sergeant Jay (1j4DDkhY91GHpOlLJgskz5)             	   ===> [1]        1  1
Searching For Albums For Jeff Thornley (0FlasGoy65q332jvE4uzOY)            	   ===> [1]        1  1
Searching For Albums For Thompson (6q6y4v50mgKaLMgHTBVuAs)                 	   ===> [1]        1  1
Searching For Albums For Thompson (3aSZ2W4GjcFPRdtMCsn7ea)                 	   ===> [1]        1  1
Searching For Albums For Starky boi (3kGGeaw2CCoyy26TEdxEJK)       

Searching For Albums For Nicoletta Scarpinella (5aJK7pLq4vftKBrc2jYmSp)    	   ===> [8]        8  8
Searching For Albums For MC Vitao CK (0zYOY8Yt3TRQV6UBDsmXMN)              	   ===> [1]        1  1
Searching For Albums For Page 44 (5HZD65s5064NBkNsGaezCe)                  	   ===> [1]        1  1
Searching For Albums For NurAllah (43nmVlrl8kAP3jZL8Dg8fE)                 	   ===> [1]        1  1
Searching For Albums For PALMY (64ykwktXeUsxwxsR1E04sa)                    	   ===> [5]        5  5
180/?      : Process [Getting Spotify Albums] Has Run For 35 Minutes and 48 Seconds.
Saving 139330 Spotify Searched For Albums
Searching For Albums For Usherwaney (1dX082e1WR8ma9srPUhq8P)               	   ===> [1]        1  1
Searching For Albums For Palomino (5gLmFC8gmj9lQslIslhFmT)                 	   ===> [1]        1  1
Searching For Albums For Truly Yours (2VbRMbuk2qrwB0qbBx1Xas)              	   ===> [6]        6  6
Searching For Albums For Modular 7even Mr.Peabody (72BfcpHlIN4BmzQGUK95YZ

...

Searching For Albums For Graham Laybourne (6kHyXXTITBosmF2RH81lYz)         	   ===> [8]        8  8
Searching For Albums For dip in the pool (6rFigrNqYr09w8qMoub2Q3)          	   ===> [32]       32  32
Searching For Albums For DeAnte Duckett & The Justified Crew (2tnTM9JFqIOWq1L7CkNejJ)	   ===> [3]        3  3
Searching For Albums For Sir (2LCsCyMrEgSg5qSWpUzv3k)                      	   ===> [8]        8  8
Searching For Albums For Avery Sunshine (2zdeiQKh54VUmz6n2CRmTn)           	   ===> [1]        1  1
2485/?     : Process [Getting Spotify Albums] Has Run For 8 Hours and 45 Minutes.
Saving 141635 Spotify Searched For Albums
Searching For Albums For Leonardo (14hEzZxTgaSdCrK99aeqii)                 	   ===> [1]        1  1
Searching For Albums For Béla Kovács (4MJZ46co2VqGiDOYLGp3Dy)            	   ===> [119]      50 . 119
Searching For Albums For Randy Ortiz (6XVpqvozhSd6poOFT5cN0J)              	   ===> [5]        5  5
Searching For Albums For David Hamilton Smith (7BS6WDWtCZwEN

Searching For Albums For Kovee (6PtjPIPv2TCC4e9gtA1e9y)                    	   ===> [2]        2  2
Searching For Albums For Elliott Carter (7IqwNdpG9USkkEH36pkKEB)           	   ===> [271]      50 .... 271
Searching For Albums For DJ FEEL-X (5w1vebuBHJxrNU9ZxBmJvj)                	   ===> [4]        4  4
Searching For Albums For Vellezerit Gashi (2M0QdHhmHBSBvpZXRKsSWk)         	   ===> [2]        2  2
Searching For Albums For Karol Kahn (7oEsLucxEfNRQLWK5u4ugo)               	   ===> [1]        1  1
2530/?     : Process [Getting Spotify Albums] Has Run For 8 Hours and 55 Minutes.
Saving 141680 Spotify Searched For Albums
Searching For Albums For Crushtomize (4INUCqM5LP7SQu5al3AJcN)              	   ===> [3]        3  3
Searching For Albums For Lorenzo Vincenzo (4ZIIBYc8JrJjx7Dt9XG915)         	   ===> [5]        5  5
Searching For Albums For Marlene Stahl (31wpIkojP52ZdrX44NFdSS)            	   ===> [1]        1  1
Searching For Albums For Anjaan Shaqs (45Fkm8EmuuzEtlAnFDs8tb)       

Searching For Albums For Yannick MYK (4o0ASqNpQernTIw4WkH5uL)              	   ===> [18]       18  18
Searching For Albums For Moby and The Void Pacific Choir (2fJQObEnbIdxhNqLHohLU4)	   ===> [47]       47  47
Searching For Albums For Cvetka Ahlin (665SZgKrQ5WNGAeOnd2Mqa)             	   ===> [69]       50  69
Searching For Albums For Alexia Langis (1KnmhAncqCZIhgi2vD9x5x)            	   ===> [8]        8  8
Searching For Albums For Bibi (5au2qk9CszVqWbFznXokWX)                     	   ===> [1]        1  1
2575/?     : Process [Getting Spotify Albums] Has Run For 9 Hours and 5 Minutes.
Saving 141725 Spotify Searched For Albums
Searching For Albums For Marie Cherrier (3hcZCdkLqQxgmLkrBbDtPB)           	   ===> [7]        7  7
Searching For Albums For envy616 (4j7rCJDZ2YMwhtaFJTx9Rp)                  	   ===> [25]       25  25
Searching For Albums For Mystic (7kYjlULjZPHd5Ge3Zu8Vhb)                   	   ===> [24]       24  24
Searching For Albums For The Crossed Swords Pipes & Drums (0B

Searching For Albums For Dario Gatto (0zAH1q9AORDzcC79Lzhm72)              	   ===> [4]        4  4
Searching For Albums For The Northern Blues Legends (7DRHyLQgMp82jetXfhYBeR)	   ===> [1]        1  1
Searching For Albums For Juliette Moraine (03f4slQW2FxnirkLZl9sBJ)         	   ===> [2]        2  2
Searching For Albums For Wilfried Schmickler, Jochen Malmsheimer (0RAnWix1UCsN1VyJ0j7dV1)	   ===> [1]        1  1
Searching For Albums For The Funky Crashers (31JlF7Gk0hVTCtfhi7HiqU)       	   ===> [14]       14  14
2620/?     : Process [Getting Spotify Albums] Has Run For 9 Hours and 14 Minutes.
Saving 141770 Spotify Searched For Albums
Searching For Albums For Sammy Smack (5RsUY7utJeJp6JpihElo21)              	   ===> [3]        3  3
Searching For Albums For Tainted Youth (3vdMUTP2KXJNJgAHXCuJfr)            	   ===> [1]        1  1
Searching For Albums For Mason Bates (3YwY9KSxS54K7cUQwifbmS)              	   ===> [43]       43  43
Searching For Albums For MAXIMUS (5rrMvGsvIGgjeBu3nlc5lF)

Searching For Albums For Denzzo (2BoWpeTmh4Rvyf3HygXmwr)                   	   ===> [5]        5  5
Searching For Albums For Pedram Azad (1sEkDexfrAocVh8pZdmJVl)              	   ===> [20]       20  20
Searching For Albums For UTP (3spodygzJKEc3EYDwFCZEf)                      	   ===> [14]       14  14
Searching For Albums For Grass Widow (5mUHvPTMAx9FxhO0z4KGS6)              	   ===> [10]       10  10
Searching For Albums For Sonal Ko Konob' (79FejheRxENbJhVVisKphR)          	   ===> [1]        1  1
2665/?     : Process [Getting Spotify Albums] Has Run For 9 Hours and 24 Minutes.
Saving 141815 Spotify Searched For Albums
Searching For Albums For D'Nar (4PCDmd3ieFPEe23KlkVArS)                    	   ===> [3]        3  3
Searching For Albums For Paula Tripodi (4NF9RXryZXGclAp2aS6q7N)            	   ===> [1]        1  1
Searching For Albums For S.H.O.K.K. (2MnujVMhxzi3sAFD1NNR0x)               	   ===> [155]      50 .. 155
Searching For Albums For Internazionale (7KGHdp8s54UBCKDCxVNQwj) 

## Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))

# Download Artist Info Data

## Determine Artists To Download

## Download Artist Info

# Download Artist Albums Data

## Determine Artists To Download

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

In [ ]:
searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = spotifyArtists[~spotifyArtists.index.isin(searchedForArtists.index)]['name']
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
from pandas import Series
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    sids = [sid for sid in sids if len(sid) == 22]
    spotifyToGet.update({sid: name for sid in sids})
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Spotify")
api  = apig.getDBAPIData("Spotify")

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
ts = timestat("Downloading Spotify Data")
#tt = termTime("today", "10:00pm")
tt = termTime("tomorrow", "10:00pm")
#tt = termTime("tomorrow", "7:00am")
#tt = termTime("today", "9:30am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistAlbumsKnown(dbID) or True:
        response = api.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistAlbums(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        print("="*150)
        api.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        api.sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

# Extra

In [ ]:
  
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)

In [ ]:
######################################################
# Juypter
######################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from ioUtils import getFile, saveFile
from webUtils import getHTML
from searchUtils import findExt
from fsUtils import setFile, isFile, fileUtil
from fileUtils import getBaseFilename
from timeUtils import timestat
from fileIO import fileIO
from dbUtils import utilsSpotifyCharts


###########################################################################
## General
###########################################################################
import urllib
from glob import glob
from time import sleep
from io import StringIO
from pandas import Series,DataFrame,concat,read_csv,date_range
from datetime import datetime
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath



######################################################
# Versions
######################################################
import sys
print("Python: {0}".format(sys.version))
import datetime as dt
start = dt.datetime.now()

print("Notebook Last Run Initiated: "+str(start))

# Global

In [ ]:
io = fileIO()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()

from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

# Spotify API

In [ ]:
# !pip install spotipy

In [ ]:
from artistSpotify import artistSpotify
#from artistIDBase import artistIDSpotify
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection


##################################################################################################################
# Base Class
##################################################################################################################
class dbSpotify:
    def __init__(self):
        self.db     = "Spotify"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistSpotify(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        self.manc = masterArtistNameCorrection()
        
        
    def getSearchDir(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return searchDir
        
    def getArtistSearchSavename(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return fileUtil(searchDir).join("{0}.p".format(psID))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
def getArtistTrackResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
            sleep(1.0)
    print(" {0}".format(len(searchResults)))
    return searchResults

In [ ]:
def getArtistAlbumResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    href           = requestResult.get('href')
    artistID       = PurePosixPath(unquote(urlparse(href).path)).parts[3]
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        sleep(3.5)
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
    print(" {0}".format(len(searchResults)))
    
    albums = [spotifyAlbumRecord(album).get() for album in searchResults]
    retval = {'href': href, 'artistID': artistID, 'albums': albums}
    return retval

In [ ]:
def getArtistSearchResults(artistName, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    sleep(0.25)
    result = sp.search(artistName, limit=limit, type='artist')
    
    artists = result.get('artists', {}) if isinstance(result,dict) else {}
    href    = artists.get('href')
    total   = artists.get('total')
    nextURL = artists.get('next')
    prevURL = artists.get('previous')
    items   = artists.get('items', [])
    retval  = [spotifyArtistRecord(item).get() for item in items]
    print(len(retval))
    
    return retval

# Artist Search

In [ ]:
from glob import glob
from masterUtils import masterUtils
mu = masterUtils()
disc = mu.getDisc("Spotify")
ts = timestat("Finding Search Files")
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
ts.stop()

In [ ]:
from fileIO import fileIO
io = fileIO()
#io.save(idata=files, ifile="spotifyFiles.p")
files = io.get("spotifyFiles.p")
len(files)

In [ ]:
from fsUtils import fileUtil, mkDir, dirUtil, moveFile
dbIO = dbSpotify()
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    artistName = fileUtil(ifile).basename
    searchDir = dbIO.getSearchDir(artistName)
    if not searchDir.exists():
        print("Making {0}".format(searchDir))
        mkDir(searchDir)
    savename = dbIO.getArtistSearchSavename(artistName=artistName)
    moveFile(ifile,savename)
    
    if (i+1) % 10000 == 0 or (i+1) == 1000 or (i+1) == 100:
        ts.update(n=i+1, N=len(files))
ts.stop()

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
ts = timestat("Getting Searched For Artists")
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
ts = timestat("Getting Artists To Download")
mmeDF = meu.getDF()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtists)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 2500:
        break
    savefile = fileUtil("spotify/artists").join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if savefile.exists():
        continue
    if nErr >= 5:
        break
    result = getArtistSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(2.5)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
    
    if (i+1) % 100 == 0:
        sleep(120)
ts.stop()

## Artist Results

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()

# Move Artist Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile

files = glob("spotify/artists/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    dst = dirUtil("/Volumes/Piggy/Discog/artists-spotify/artists").join(src.name)
    if dst.exists():
        continue
    moveFile(src.path, dst)

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

# Move Album Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile,mkDir
from artistModValue import artistModValue
amv = artistModValue()

files = glob("spotify/albums/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    albumsData = io.get(ifile)
    artistID = albumsData['artistID']
    modValue = amv.getModVal(artistID)
    modDir = dirUtil("/Volumes/Piggy/Discog/artists-spotify/").join(str(modValue))
    if not modDir.exists():
        mkDir(modDir)
    dst = dirUtil(modDir).join("{0}.p".format(artistID))
    if dst.exists():
        continue
    print("  ==>  {0}".format(dst))
    moveFile(src.path, dst)

# Create Master Artist ID List

In [ ]:
from glob import glob
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
artistsData = []
N = len(files)
ts = timestat("Loading {0} Spotify Artist Search Files".format(N))
for i,ifile in enumerate(files):    
    artistsData += io.get(ifile)
    if (i+1) % 5000 == 0 or (i+1) == 1000:
        ts.update(n=i+1, N=N)
ts.stop()

In [ ]:
def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

df = DataFrame(artistsData)
df["followers"] = df["followers"].apply(getFollowers)
df.index = df['sid']
df.drop(['sid'], axis=1, inplace=True)
df = df[~df.index.duplicated(keep='first')]
df = df.sort_values(by='popularity', ascending=False)

print("Saving {0} Spotify Artists Data".format(df.shape[0]))
io.save(idata=df, ifile="/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
df.head(10)

In [ ]:
print(df.shape)

# Artist Albums

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

## Download Albums Data

In [ ]:
spotifyArtists['popularity'].hist(bins=100, log='y')

In [ ]:
spotifyArtists[spotifyArtists["popularity"] >= 60].shape

In [ ]:
knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
spotifyToGet = Series(spotifyToGet)
#spotifyToGet = knownSpotify["ArtistName"]
#spotifyToGet.index = list(knownSpotify["Spotify"].values)
#spotifyToGet.name = "name"
#spotifyToGet

In [ ]:
toget = {}
from artistModValue import artistModValue
from masterUtils import masterUtils
from fsUtils import dirUtil, mkDir
mu   = masterUtils()
amv  = artistModValue()
disc = mu.discs["Spotify"]


#spotifyToGet = spotifyArtists[spotifyArtists["popularity"] >= 60]
#print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
#for i,(artistID,artistName) in enumerate(spotifyToGet["name"].iteritems()):

print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
nKnown = 0
for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
    modVal   = amv.getModVal(artistID)
    modDir   = dirUtil(disc.getArtistsDir()).join(str(modVal))
    if not modDir.exists():
        print("Making {0}".format(modDir))
        mkDir(modDir)
    savefile = dirUtil(modDir).join("{0}.p".format(artistID))
    if savefile.exists():
        nKnown += 1
        continue
    #print("{0: <25}{1: <20} ({2})".format(artistName,artistID,modVal))
    toget[savefile] = (artistName,artistID)
    if len(toget) >= 2500:
        break
print("Found {0} Known Artists".format(nKnown))
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(7.5)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(15.0)
        print("")
    
    if (i+1) % 25 == 0:
        sleep(30.0)
ts.stop()

## Determine Artists To Download

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
io.save(idata={}, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
stop = 150
dbIO = dbSpotify()
api  = spotifyAPIIO()
ts   = timestat("Getting Artist Data From Spotify API")
n    = 0

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    savename = dbIO.getArtistAlbumsSavename(artistID)
    if savename.exists():
        searchedForArtists[artistID] = artistName 
        continue
        
    #print(artistID,artistName,savename)
    response = api.getArtistAlbums(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistAlbums(artistID=artistID, artistAlbums=response)
    searchedForArtists[artistID] = artistName    
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Error Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
if False:
    searchedForArtists = io.get("spotifySearchedForArtists.p")
    print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

    ts = timestat("Finding Spotify Saves")
    first = True
    for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
        if searchedForArtists.get(artistID) is not None:
            continue
        if first:
            print("Start: {0}".format(i))
            first = False
        if dbIO.getArtistAlbumsSavename(artistID).exists():
            searchedForArtists[artistID] = artistName
            if len(searchedForArtists) % 5000 == 0:
                break
        else:
            break

        if (i+1) % 100 == 0:
            ts.update(n=i+1,N=len(spotifyToGet))        
            #io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

    ts.stop()
    print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
    io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

In [ ]:
spotifyArtists.head()

In [ ]:
from dbArtistsBase import dbArtistsBase
from fileUtils import getBaseFilename
from fsUtils import isFile, setFile, setDir
from ioUtils import getFile, saveFile
from timeUtils import timestat
from sys import prefix
import urllib
from time import sleep
from webUtils import getHTML
from pandas import Series
    
from fileIO import fileIO
from fsUtils import fileUtil

In [ ]:
from dbArtistsParse import dbArtistsSpotifyAPI
from dbArtistsSpotify import dbArtistsSpotify
dbAP = dbArtistsSpotifyAPI(dbArtistsSpotify())

In [ ]:
for modVal in range(100):
    dbAP.parse(modVal=modVal, expr='< 0 Days', force=False)
    
#for modVal in range(100):
#    dbAP.parseSearch(modVal, expr='< 0 Days', force=False)

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()
modVal = 91
idx = artistSearchData.reset_index()['sid'].apply(amv.getModVal) == modVal
idx.index = artistSearchData.index
artistSearchData[idx]

In [ ]:
artistSearchData.head()

In [ ]:
art = artistSpotify()
art.getData(inputData).show()

In [ ]:
io.get("/Users/tgadfort/Music/Discog/artists-spotify-db/91-DB.p").get('1uNFoZAHBGtllmzznpCI3s').show()

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
mmeDF = meu.getDF()

In [ ]:

toget = {}
for i,(artistID,artistName) in enumerate(spotifyArtists["name"].iteritems()):
    if i >= 5:
        break
    savefile = fileUtil("spotify/albums").join("{0}--{1}.p".format(artistID,manc.clean(artistName)))
    if savefile.exists():
        pass
        #continue
    toget[savefile] = (artistName,artistID)
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(3.0)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

## Artist Albums Results

In [ ]:
from glob import glob
files = glob("spotify/albums/*.p")
artistAlbumsData = {}
for ifile in files:
    albumsData   = io.get(ifile)
    artistID     = albumsData['artistID']
    artistAlbums = albumsData['albums']
    if artistAlbumsData.get(artistID) is not None:
        raise ValueError("Already have data for {0}".format(artistID))
    artistAlbumsData[artistID] = artistAlbums

In [ ]:
retval = getArtistAlbumResults(artistID='46sPgFJpEcoxUJNr1ouR9G', artistName='dummy')

In [ ]:
s = Series(artistAlbumsData)

In [ ]:
s.apply(len)

In [ ]:
retval = sp.artist_albums('13y7CgLHjMVRMDqxdx0Xdo', limit=10, offset=0)

In [ ]:
retval['href']

In [ ]:
href='https://api.spotify.com/v1/artists/13y7CgLHjMVRMDqxdx0Xdo/albums?offset=0&limit=10&include_groups=album,single,compilation,appears_on'
href

In [ ]:
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath

url = 'http://www.example.com/hithere/something/else'

PurePosixPath(unquote(urlparse(href).path)).parts[3]

In [ ]:
from urlparse import urlparse, parse_qs
s = "https://xx.com/question/index?qid=2ss2830AA38Wng"
parse_qs(urlparse(s).query)['qid'][0]

In [ ]:
rParen = r'\((.*?)\)+'
rFeat  = r'\b(feat.\s|Feat.\s|with\s)\b'
rSuffix= r'\s-\sRemix'

In [ ]:
featureValue = regex.search(rFeat, parenValue.group())


In [ ]:
retval = sp.artist_top_tracks('60d24wfXkVzDSfLS6hyCjZ')

In [ ]:
help(sp.tracks)

In [ ]:
requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)


In [ ]:
DataFrame(retval['items']).T[0]

In [ ]:
retval2 = sp.artist_albums('60d24wfXkVzDSfLS6hyCjZ', limit=50, offset=450)

In [ ]:
retval2['previous']

In [ ]:

results = sp.search(q='weezer', limit=20)
for idx, track in enumerate(results['tracks']['items']):
    print(idx, track['name'])

In [ ]:
result.keys()

In [ ]:
urn = 'spotify:artist:3jOstUTkEu2JkjvRdBA5Gu'

#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())

artist = sp.artist(urn)
artist

In [ ]:
result['artists'].keys()

In [ ]:
help(sp.search)

In [ ]:
search_str = 'Radiohead'
result = sp.search(search_str)
pprint.pprint(result)